# ZIPA debugging

In [1]:
from tira_kws.models.zipa import load_zipa_large_transducer, decode_transducer, transducer_params
from tira_kws.models.zipa import load_zipa_large_crctc, decode_ctc, ctc_params
from lhotse.features.kaldi.extractors import Fbank, FbankConfig
import numpy as np

/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1306: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, x, y, alpha):
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1315: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad):
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1512: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(
/home/mjsimmons/projects/zipa/zipformer_crctc/scaling.py:1551: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  def backward(ctx, ans_grad: Tensor):


## ZIPA
Instantiate model, feature extractor and tokenizer

In [2]:
tdcr = load_zipa_large_transducer()
ctc = load_zipa_large_crctc()

tdcr.to('cuda')
ctc.to('cuda')

type(tdcr), type(ctc)

(model.AsrModel, model.AsrModel)

In [3]:
fbank = Fbank(FbankConfig(num_filters=80, dither=0.0, snip_edges=False))
fbank

In [4]:
import sentencepiece
from tira_kws.constants import ZIPA_SENTENCEPIECE_MODEL

tokenizer = sentencepiece.SentencePieceProcessor()
tokenizer.load(str(ZIPA_SENTENCEPIECE_MODEL))

True

## Load LibriSpeech audio

In [5]:
import os
from pathlib import Path
import random
import librosa
import nava
import IPython.display as ipd

In [7]:
data_dir = Path(os.environ.get("DATASETS"))
librispeech_dir: Path = data_dir/"LibriSpeech"
dev_audio = librispeech_dir/"dev-clean"
subdir = random.choice(list(dev_audio.glob("*")))
sub_subdir = random.choice(list(subdir.glob("*")))
audio_file = random.choice(list(sub_subdir.glob("*.flac")))
audio_file

PosixPath('/mnt/LocalStorage/mjsimmons/datasets/LibriSpeech/dev-clean/5338/24615/5338-24615-0001.flac')

In [8]:
wav, sr = librosa.load(audio_file, sr=16_000)
wav.shape, sr

((121680,), 16000)

In [9]:
ipd.Audio(wav, rate=sr)

In [10]:
features = fbank.extract_batch([wav], sampling_rate = sr)
len(features), features[0].shape

(1, (761, 80))

In [11]:
feature = features[0]
feature_lens = np.array([feature.shape[1]], dtype=np.int64)

## Load k2 dataset

In [12]:
from lhotse.recipes import prepare_librispeech
from lhotse.dataset import K2SpeechRecognitionDataset, DynamicBucketingSampler, OnTheFlyFeatures
from lhotse.cut import CutSet
from torch.utils.data import DataLoader

In [13]:
ls_cuts = prepare_librispeech(librispeech_dir)
ls_cuts

Dataset parts:   0%|          | 0/3 [00:00<?, ?it/s]

Distributing tasks: 0it [00:00, ?it/s]

Processing:   0%|          | 0/2620 [00:00<?, ?it/s]

Distributing tasks: 0it [00:00, ?it/s]

Processing:   0%|          | 0/28539 [00:00<?, ?it/s]

Distributing tasks: 0it [00:00, ?it/s]

Processing:   0%|          | 0/2703 [00:00<?, ?it/s]

{'test-clean': {'recordings': RecordingSet(len=2620),
  'supervisions': SupervisionSet(len=2620)},
 'train-clean-100': {'recordings': RecordingSet(len=28539),
  'supervisions': SupervisionSet(len=28539)},
 'dev-clean': {'recordings': RecordingSet(len=2703),
  'supervisions': SupervisionSet(len=2703)}}

In [14]:
dev_cuts = CutSet.from_manifests(
    recordings=ls_cuts['dev-clean']['recordings'],
    supervisions=ls_cuts['dev-clean']['supervisions'],
)

In [15]:
k2_ds = K2SpeechRecognitionDataset(return_cuts=True, input_strategy=OnTheFlyFeatures(fbank))
sampler = DynamicBucketingSampler(dev_cuts, max_duration=30.0, max_cuts=128, shuffle=False)

In [16]:
dl = DataLoader(k2_ds, batch_size=None, sampler=sampler, num_workers=0)
dl

In [17]:
batch = next(iter(dl))

In [18]:
decode_ctc(params=ctc_params, model=ctc, sp=tokenizer, batch=batch)

{'ctc-greedy-search': [['̥ɖyɖyǀh̥hɖy<sos/eos>ʘ<sos/eos>ʘʒryʌ̴y̺yʌ̺ytyʒyɻyʌʘkyrkʘyʌyʌ̺ʌrʌtat̪<sos/eos>ɭ̥h̥ɐ̥kak̪trtʌ̪ʌʘhʘyhʂkœkʘ<sos/eos>œhʘ<sos/eos>ʘ<sos/eos>ɖʘ<sos/eos>̥ɖʘʄ̥ʘⱱ̥ɖ<sos/eos>̥yʌyʌyʘyrhɭyɭʘ<sos/eos>ryakyʌkakʌ̴ʒarkrkryryr̺raʌrkarɜrkrɬrʘh<sos/eos>ʘktkrkʘryrʌyarhrʐrʘh<sos/eos>r<sos/eos>h<sos/eos>ɘʂ<sos/eos>hʘiʂʘ̥ʄ̥yk̥i̥ʄⱱ̥ɾⱱ<sos/eos>ʘ<sos/eos>kⱱʘ<sos/eos>ʘ̥<sos/eos>ʘryr̴tʐrtʌ̺kakayʌyʌyʌɭʌh<sos/eos>h<sos/eos>hʂhɖy<sos/eos>ʂʘ<sos/eos>ʂ<sos/eos>hiʘ̥ʘ̥ʘh̴rʘktʌœrɖrvrʌ̺akrʌr̴̩ʌ̴aʌ̺ʌtɜrtʌtrtʌ̺ʌɖhʘɖǀɖʘǀkʘhɖʁʄʘiy̥ʄ̥ʘ̥ɖ<sos/eos>k̥ⱱʘʄh̥ryⱱ̥rⱱʘɖrykrʘʒɭkʄrʁky̥ɾ̥ʄ̥yʘœykyrkrktʌr<sos/eos>h<sos/eos>hyayk̺kʘ<sos/eos>kʘyrkœkrʘkrkɐrkrt̺ʌaʃʌ̺ʘ̺kr̺rwtʂkrkɭk̪tœkɖkrkayɭʌ<sos/eos>ʂʘktkɖk̪k̥ɖʘʏ̥ʘɖ̥k̥ʘ̥ɖʘⱱʘʌyʌ̪ʌyrtrtraʌyʌʘyʘhyʘyʘrtr̚rhr̺yrʒyrʘrarʘʐʘ<sos/eos>ʄʘ<sos/eos>ʘ̪<sos/eos>ɖ̥k̥ʘʄʘ̥ⱱɖkɖ̥ɖ<sos/eos>̥<sos/eos>ɾkʘʄʘ<sos/eos>kyʘykʌkʘrkrʒa̴krʌk̪kœk<sos/eos>k<sos/eos>ʘʂʘⱱ̥ykɖ̪̥ɖʄ'],
  ['<sos/eos>ky̥ʘ̥<sos/eos>ykrkrt̺̩ryɻyatʌ̺trara̪ʌayk̪ktrtœkyʘyʒʌ̺kʌkrkœ<sos/eos>hʘhⱱhyɰrkykryrkrkryrœʐrk̪rⱱʐkʄɖ̥ɐʄ̥ʘhⱱʘʒⱱry

In [19]:
decode_transducer(params=transducer_params, model=tdcr, sp=tokenizer, batch=batch)

{'greedy_search': [['ɒkɒɒɒɒɒɒeχʝʝʌʝʌʌjjjjħħħɦɳʝeʝʌʌʌʌǁǁǁǁǁǁǁǁeʝeʝχχɒɒʟʟkkkkkkkkkkkkkkeeʝʌɒkχχɒɒkɒkkχχʌɦeχʌʌʌʌʝjʜʔɞɞɞɞkkkkʔǁǁǁǁǁǁǁeeʌʌɦǃʌʌʌʌɒɒɒɒɒɒkkkkkkkkɒɒkɒɒkkkeeeeʌʌʌʝʝʌʝʝʌʌχɒɒɒɒɒkɒkkkkkkeeʌʌʔʔʔʔʝʝχχeʝʌʌʝʝʝʝʝʌʝʝʝkkkɒɒkkkkkkkkkkkkkkɒkkkkkkkǁǁǁǁǁǁǁǁʌʌʌʌɦɦʝʝkɒɒɒχχʌʌɒɒkǃǁǁǁǁkkkkʝeʝeʌʌʌʌɒɒkχǁǁǁǁeeeeeekχɰǁʟkkkkkkkkkkkkkʝeʌʌʌʌʝʌʌʌʌʌɒɒʌʌʌʌɒɒeɞɞɞllǃǃkkkkkkkkkkkkkkkkeeʌʌʌʌeeʝʝʝʝǁǁǁǁkkkkkkkkkkk'],
  ['kkkkkkkkʝʝʝʝʌʌʌʌʌʝʝǁǁǁʌʌʌʌɒkɱɱeʝʌʝʝʌǁǁǁǁɒɒɒɒʌʌʌjħħʔʔʟʔʟǃjʌʌʌʔkkkkkkkɒɒɒɒʔʔʔʔllmmɒɒkɒkkkkmmʝʌʝʝʌʌʌʌɦǁǁǁǁǁʌʌʌʌʔʔkkǁǁʌʌʝjʌʌɒɒχχħħħħʌʌʌʌɒkkǁħkħħɒɒǃǃǁǁǁǁkkkkkkkkʔǁǁǁʔʔʔʔɨʔlʔʌʌɒkkʝʌʌɒkɒǃkkɰʌʟʟkkɒɒkɒɒkɒkkkkkkkkkɒɒɒɒkǃʔʔkkkkǃǃǃǃʝʝʌʌkkʔʔʝʝʌʌɒkɒɒɒɒkɒkkkkkkkkkkkkkkɒkχχkkkkkkeeʝeʌʌɒɒʌʝʔʌʝʝʌʌʌʌʝʌʝʝʌʌǁǁǁǁǁǁǁǁǁǁǁǁeʝʝʝʌʌʌʌʝeʝeʌʌʌʌɒɒkkʔʌʌʔʌʌɒɒkkɒɒkkkk']]}